### Step 1: Create Your Local FunctionDefine the actual Python function you want to run. This is the code that will execute once the AI tells you what parameters to use

In [31]:
import requests


def get_current_weather(city_name: str) -> dict:
    """
    Fetches the current weather data for a given city using OpenWeatherMap API.

    :param city_name: Name of the city (e.g., 'London' or 'New York')
    :param api_key: Your OpenWeatherMap API secret key
    :return: A dictionary containing temperature, humidity, and weather description
    """
    url = "https://api.openweathermap.org/data/2.5/weather"

    query_parameters = {
        "q": city_name,
        "appid": "a2b71c0cffd8ffe14e6dfa33febd04cc",
        "units": "metric",
    }

    try:
        response = requests.get(url, params=query_parameters, timeout=10)
        response.raise_for_status()
        data = response.json()

        return {
            "city": data["name"],
            "temperature": f"{data['main']['temp']}°C",
            "humidity": f"{data['main']['humidity']}%",
            "description": data["weather"][0]["description"].title(),
        }

    except requests.exceptions.HTTPError as http_err:
        return {"error": f"HTTP error occurred: {http_err}"}
    except requests.exceptions.RequestException as req_err:
        return {"error": f"Network error occurred: {req_err}"}
    except KeyError:
        return {"error": "Unexpected data format received from the API."}

### Define another function to add two numbers


In [53]:
def add(a: int, b: int) -> int:
    """
    Adds two integers and returns the result.

    :param a: First integer
    :param b: Second integer
    :return: The sum of a and b
    """
    return a + b

In [32]:
import os
from dotenv import load_dotenv
import json
from openai import OpenAI

import textwrap


def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception:
        print(text)


load_dotenv('/Users/dineshjadhav/Desktop/genai-course/openai_key.env', override=True)

api_key = os.getenv("OPENAI_API_KEY")
print("present:", bool(api_key))
print("length:", len(api_key) if api_key else None)
print("prefix:", api_key[:7] if api_key else None)
print("suffix:", api_key[-4:] if api_key else None)

pretty_print("API key loaded successfully.")
client = OpenAI(api_key=api_key)

present: True
length: 164
prefix: sk-proj
suffix: H58A
API key loaded successfully.


In [33]:
resp_dev = client.responses.create(
        model="gpt-4o-mini",
        instructions="you are weather expert",
        input=[
            {"role": "user", "content": "What is temperature in London?"}
        ]
    )
print(resp_dev.output_text)


I can't provide real-time weather updates. For the current temperature in London, please check a reliable weather website or app.


### Step 2: Define the Tool Schema for OpenAIYou must describe your function to OpenAI using a JSON schema. This tells the AI what your function does, what arguments it expects, and which ones are required.

In [47]:
weather_tool_definition = {
    "type": "function",
    "function": {
        "name": "get_current_weather",
        "description": "Get the current weather for a specific city location",
        "parameters": {
            "type": "object",
            "properties": {
                "city_name": {
                    "type": "string",
                    "description": "The city name, e.g., Mumbai, London, Tokyo",
                }
            },
            "required": ["city_name"],
        },
    },
}

add_tool_definition = {
    "type": "function",
    "function": {
        "name": "add",
        "description": "Add two integers and return the result",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "integer",
                    "description": "The first integer to add",
                },
                "b": {
                    "type": "integer",
                    "description": "The second integer to add",
                },
            },
            "required": ["a", "b"],
        },
    },
}


### Step 3: Pass the Tool Definition to OpenAISend the user's prompt along with your tools list to the chat completions endpoint. If the user asks a question requiring weather data, the model will decide to call your function

In [48]:
messages = [{"role": "user", "content": "What is five plus eight?"}]

# Call the OpenAI API with tools enabled
response = client.chat.completions.create(
    model="gpt-4o-mini",  # or gpt-4o-mini
    messages=messages,
    tools=[weather_tool_definition,add_tool_definition],
    tool_choice="auto" 
)

response_message = response.choices[0].message
tool_calls = response_message.tool_calls
print("Model response:", response_message.content)


Model response: None


In [55]:
if tool_calls:
    # OpenAI requires you to append the model's tool request to the message history first
    tool_messages = []

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        
        if function_name == "get_current_weather":
            # Extract arguments generated by the model
            function_args = json.loads(tool_call.function.arguments)
            
            # Execute your local function
            function_response = get_current_weather(
                city_name=function_args.get("city_name")
            )
            
            tool_messages.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": function_name,
                "content": json.dumps(function_response),
            })
        elif function_name == "add":
            # Extract arguments generated by the model
            function_args = json.loads(tool_call.function.arguments)
            
            # Execute your local function
            function_response_add = add(
                a=function_args.get("a"),
                b=function_args.get("b")
            )
            
            tool_messages.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": function_name,
                "content": json.dumps({"result": function_response_add}),
            })
    # Call OpenAI one final time with the function results included
    final_messages = [messages[0], response_message] + tool_messages
    final_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=final_messages
    )
    
    print(final_response.choices[0].message.content)
else:
    # If the user asked a normal question that didn't need the weather tool
    print(response_message.content)


Five plus eight equals thirteen.


# LLM Function Calling Sequence of Operations

This document outlines the exact execution lifecycle when an LLM utilizes external tools to answer a user prompt.

## 1. Sequence Flowchart

```text
[ User Prompt ] 
       │
       ▼
1. LLM Evaluation ───( Has Data? )───► Yes ──► [ Direct Response ]
       │
       ▼ No
2. Schema Matching (Scans tools list for the best match)
       │
       ▼
3. Parameter Extraction (Resolves arguments into structured JSON)
       │
       ▼
4. Interception & Execution (Backend stops LLM, runs your code)
       │
       ▼
5. Context Feeding (Function output appended to conversation history)
       │
       ▼
6. Final Synthesis (LLM reads raw output and writes natural response)
```

---

## 2. Detailed Operational Breakdown

### Step 1: LLM Evaluation & Gap Detection
* **Action**: The LLM receives the user prompt (e.g., *"What is the weather in Mumbai?"*).
* **Logic**: The LLM evaluates its internal training data. Because live weather changes constantly, the LLM determines it lacks this real-time data.

### Step 2: Tool Schema Matching
* **Action**: Instead of giving up or hallucinating, the LLM scans the list of `tools` (the JSON schemas) provided in the API call.
* **Logic**: It compares the user's intent against the `description` fields in your schemas. It identifies that `get_current_weather` is the correct tool to bridge its knowledge gap.

### Step 3: Parameter Resolution
* **Action**: The LLM parses the user's natural language to extract required variables.
* **Logic**: It maps *"Mumbai"* to the `city_name` property defined in your schema. It outputs a specific instruction payload (`tool_calls`) containing a unique ID, the function name, and the structured arguments: `{"city_name": "Mumbai"}`.

### Step 4: Backend Interception & Execution
* **Action**: The OpenAI cloud **stops processing** and pauses. It sends this JSON payload back to your backend server application.
* **Logic**: Your code intercepts this payload, reads the arguments, and executes your local code (`get_current_weather("Mumbai")`). This is where your code actually talks to the live Weather API.

### Step 5: Context Feeding (The Loop Back)
* **Action**: Your local function receives the raw data back from the Weather API (e.g., `{"temp": 30, "condition": "Sunny"}`).
* **Logic**: Your application packages this raw data into a new message with the role `"tool"`, linking it explicitly to the unique `tool_call_id` generated in Step 3. You append this to the conversation history.

### Step 6: Final Synthesis
* **Action**: Your backend sends the entire updated conversation history (User query + LLM tool request + Your function's raw data output) back to OpenAI for a second API call.
* **Logic**: The LLM reads the raw weather data, translates it into a friendly, natural sentence (e.g., *"It is currently 30°C and sunny in Mumbai."*), and delivers it to the end user.
